# 22. heapq and bisect

After this notebook you'll be able to:

- always get the smallest (or biggest) item out fast with **`heapq`**,
- keep the k biggest items as numbers arrive,
- find where a value belongs in a **sorted** list with **`bisect`**, and do the same search by hand.

Two small tools from Python's standard library. Neither needs installing; you `import` them.

## Part 1: `heapq`, a smallest-first pile

### A hospital emergency room

At a ticket counter, people are served in the order they arrived (the queue from notebook 21). In a hospital emergency room, they aren't. Each patient gets an urgency number, and whoever is most urgent is seen next, whenever they arrived.

A **heap** works like that. You can put items in at any time, and when you take one out, you always get the **smallest**. Python's heap tools live in a module called **`heapq`** ("heap queue").

`import heapq` brings in the whole module, so you write its name in front of each tool: `heapq.heappush(...)`.

In [ ]:
import heapq

heap = []
heapq.heappush(heap, 5)
heapq.heappush(heap, 1)
heapq.heappush(heap, 8)
heapq.heappush(heap, 3)
print(heap)
print(heap[0])
print(heapq.heappop(heap))
print(heapq.heappop(heap))
print(heap)

```
[1, 3, 8, 5]
1
1
3
[5, 8]
```

A heap is a **normal Python list**. You start with an empty one and use `heapq`'s tools on it:

| What | How |
| --- | --- |
| put an item in | `heapq.heappush(heap, x)` |
| take the smallest out | `heapq.heappop(heap)` |
| look at the smallest, without taking it | `heap[0]` |
| how many items | `len(heap)` |

### It looks unsorted. That's fine.

Look at the first `print`: `[1, 3, 8, 5]`. That's not sorted (8 comes before 5). A heap doesn't keep the whole list in order, because that would be slow. It keeps only enough order to promise one thing: **`heap[0]` is always the smallest**. Each push or pop moves a few items around to keep that promise.

So `heap[1]` or `heap[-1]` don't mean anything. Only `heap[0]`, and only what `heappop` gives you.

Pop everything, and the items come out in sorted order. (`end=" "` makes `print` put a space at the end instead of a new line, so the numbers stay on one line.)

In [ ]:
heap = []
for n in [5, 1, 8, 3, 9, 2]:
    heapq.heappush(heap, n)
print(heap)

while heap:
    print(heapq.heappop(heap), end=" ")

```
[1, 3, 2, 5, 9, 8]
1 2 3 5 8 9
```

### `heapify`: turn a whole list into a heap

If you already have a list, `heapq.heapify` rearranges that list into a heap in one go:

In [ ]:
nums = [7, 2, 9, 4, 1]
heapq.heapify(nums)
print(nums)
print(nums[0])

```
[1, 2, 9, 4, 7]
1
```

Like `sort()`, `heapify` changes the list in place and returns `None`. So write `heapq.heapify(nums)` on its own line, never `nums = heapq.heapify(nums)`.

### Biggest first: the minus trick

`heapq` only does smallest-first. To get the **biggest** first, store every number with a minus sign. The biggest number becomes the smallest negative, so it comes out first. Put the minus back when you take it out.

**Predict first:** what are the three numbers printed at the end?

In [ ]:
heap = []
for n in [7, 2, 9, 4, 1]:
    heapq.heappush(heap, -n)
print(heap)
print(-heapq.heappop(heap))
print(-heapq.heappop(heap))
print(-heap[0])

```
[-9, -4, -7, -2, -1]
9
7
4
```

```
 numbers:        7    2    9    4    1
 stored as:     -7   -2   -9   -4   -1
 smallest stored is -9  ->  out it comes  ->  put the minus back: 9
```

Two places to remember the minus: when you push (`-n`), and when you take out (`-heapq.heappop(heap)`).

### Tuples: `(priority, item)`

In the emergency room you want to know **who** is next, not only their urgency number. Push a tuple: the number first, then the thing it belongs to.

In [ ]:
patients = []
heapq.heappush(patients, (2, "Riya"))
heapq.heappush(patients, (1, "Aman"))
heapq.heappush(patients, (3, "Tanvi"))
heapq.heappush(patients, (1, "Kabir"))

while patients:
    urgency, name = heapq.heappop(patients)
    print(urgency, name)

```
1 Aman
1 Kabir
2 Riya
3 Tanvi
```

Tuples are compared item by item, from the left. Python compares the first items; only if they're equal does it look at the second ones. So the heap sorts by urgency, and a tie (Aman and Kabir, both 1) is broken by the name, alphabetically.

### The k biggest or smallest: `nlargest` and `nsmallest`

When you need the few biggest or smallest items once, there's a shortcut:

In [ ]:
scores = [55, 92, 78, 99, 63, 85]
print(heapq.nlargest(3, scores))
print(heapq.nsmallest(2, scores))

```
[99, 92, 85]
[55, 63]
```

### Keeping only the k biggest, as numbers arrive

Sometimes the numbers arrive one at a time, and after each one you need to know "what's the 3rd biggest so far?".

Think of a top-3 leaderboard with only 3 slots. When a new score arrives, put it on the board. If there are now 4, remove the **smallest** one. The board always holds the 3 biggest so far, and the smallest score on the board is the 3rd biggest overall.

A heap of size k does exactly that, and its smallest item, `heap[0]`, is the k-th biggest. Run it and watch the board:

In [ ]:
k = 3
heap = []
for score in [5, 1, 9, 3, 7, 8]:
    heapq.heappush(heap, score)
    if len(heap) > k:
        heapq.heappop(heap)
    print("after", score, "-> board", sorted(heap), "  3rd biggest so far:", heap[0])

```
after 5 -> board [5]   3rd biggest so far: 5
after 1 -> board [1, 5]   3rd biggest so far: 1
after 9 -> board [1, 5, 9]   3rd biggest so far: 1
after 3 -> board [3, 5, 9]   3rd biggest so far: 3
after 7 -> board [5, 7, 9]   3rd biggest so far: 5
after 8 -> board [7, 8, 9]   3rd biggest so far: 7
```

(`sorted(heap)` is only there to make the board easy to read. For the first two scores there aren't 3 yet, so `heap[0]` is only the smallest so far.)

When 3 arrived, the board was `[1, 5, 9]` plus 3: four scores, so the smallest, 1, was thrown away. A small score like 1 never stays long on a top-3 board.

On LeetCode this shows up in a design class (notebook 19), with the heap kept on `self` between calls. That's Task 7.

## Part 2: `bisect`, finding a place in a sorted list

### Putting a book on a sorted shelf

Your books are in order of height. You get a new book. You don't check every book; you look in the middle, see whether the new one is taller or shorter, and keep halving the part of the shelf you look at. In a few looks you find its place.

That's what **`bisect`** ("cut in two") does for a **sorted** list: it tells you the index where a value **is**, or where it **would go**.

In [ ]:
import bisect

nums = [10, 20, 30, 30, 40]
print(bisect.bisect_left(nums, 30))
print(bisect.bisect_right(nums, 30))
print(bisect.bisect_left(nums, 25))
print(bisect.bisect_left(nums, 5))
print(bisect.bisect_left(nums, 50))

```
2
4
2
0
5
```

Picture the gaps between the boxes. `bisect` tells you which gap the value belongs in:

```
 index:        0      1      2      3      4
            ┌──────┬──────┬──────┬──────┬──────┐
 nums:      │  10  │  20  │  30  │  30  │  40  │
            └──────┴──────┴──────┴──────┴──────┘
 gap:      0      1      2      3      4      5
                         ▲             ▲
            bisect_left(30) = 2        bisect_right(30) = 4
            (before the 30s)           (after the 30s)

 25 isn't there: it would go in gap 2, between 20 and 30.
  5 would go first (gap 0). 50 would go last (gap 5, which is len(nums)).
```

- **`bisect_left(nums, x)`**: the place for `x` **before** any items equal to `x`. Another way to say it: the index of the first item that is `>= x`.
- **`bisect_right(nums, x)`**: the place **after** any items equal to `x`. The index of the first item that is `> x`.
- If `x` isn't in the list, both give the same answer.

Three handy uses, all on the same `nums`:

| Question | Code | Answer for `x = 30` |
| --- | --- | --- |
| How many items are smaller than `x`? | `bisect_left(nums, x)` | 2 |
| How many times does `x` appear? | `bisect_right(nums, x) - bisect_left(nums, x)` | 4 − 2 = 2 |
| Is `x` in the list? | `i = bisect_left(nums, x)`, then `i < len(nums) and nums[i] == x` | `True` |

For the last one, both checks matter: `i` can be `len(nums)` (past the end), and `and` stops before `nums[i]` would crash.

**`bisect.insort(nums, x)`** finds the place and inserts `x` there, so the list stays sorted.

**Careful:** `bisect` trusts that the list is already sorted. On an unsorted list it doesn't complain; it gives you a wrong answer.

### The same thing, by hand: binary search

What `bisect_left` does inside is a **binary search**: look at the middle, throw away the half that can't hold the answer, repeat. LeetCode often asks you to write it yourself, so here it is, with a `print` so you can watch it.

`left` and `right` mark the part of the list still worth looking at. They're two pointers, like in notebook 13, but they jump to the middle instead of stepping one at a time.

In [ ]:
def my_bisect_left(nums, x):
    left = 0
    right = len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        print(f"  left={left} right={right} mid={mid} nums[mid]={nums[mid]}")
        if nums[mid] < x:
            left = mid + 1      # the answer is to the right of mid
        else:
            right = mid - 1     # mid might be the answer; look left of it too
    return left

nums = [10, 20, 30, 30, 40, 50, 60]
print(my_bisect_left(nums, 30))

```
  left=0 right=6 mid=3 nums[mid]=30
  left=0 right=2 mid=1 nums[mid]=20
  left=2 right=2 mid=2 nums[mid]=30
2
```

Step by step, looking for 30:

```
 index:   0   1   2   3   4   5   6
 nums:   10  20  30  30  40  50  60

 left=0 right=6 mid=3   nums[3]=30, not < 30   -> right = 2   (look in 0..2)
 left=0 right=2 mid=1   nums[1]=20, < 30       -> left = 2    (look in 2..2)
 left=2 right=2 mid=2   nums[2]=30, not < 30   -> right = 1
 left=2 > right=1: stop.  The answer is left = 2.
```

- **`mid = (left + right) // 2`** is the middle index. It must be `//`, because an index can't be 2.5.
- Each step throws away half of what's left. For a list of a million items, that's only about 20 steps.
- When the loop stops, `left` has landed on the answer: the first index whose item is `>= x`.

**Predict first:** 45 isn't in the list. Where would it go? Then run it.

In [ ]:
print(my_bisect_left(nums, 45))
print(bisect.bisect_left(nums, 45))

```
  left=0 right=6 mid=3 nums[mid]=30
  left=4 right=6 mid=5 nums[mid]=50
  left=4 right=4 mid=4 nums[mid]=40
5
5
```

Index 5, between 40 and 50. The loop and `bisect` agree.

## Common mistakes

**Thinking `heap[-1]` is the biggest.**

In [ ]:
heap = []
for n in [5, 1, 8, 3]:
    heapq.heappush(heap, n)
print(heap[-1])

```
5
```

The biggest is 8, but `heap[-1]` is 5. Only `heap[0]` has a meaning. For biggest-first, use the minus trick.

**Popping from an empty heap.**

In [ ]:
heap = []
heapq.heappop(heap)

```
IndexError: index out of range
```

Check `if heap:` (or `while heap:`) before popping.

**Using `/` for the middle.**

In [ ]:
nums = [10, 20, 30, 40]
mid = (0 + 3) / 2
print(nums[mid])

```
TypeError: list indices must be integers or slices, not float
```

`/` always gives a decimal (`1.5`), and an index must be a whole number. Use `//`.

**Forgetting `heapq.` in front.** With `import heapq`, every tool is written `heapq.something`. Plain `heappush(heap, 1)` gives `NameError: name 'heappush' is not defined`.

---

# ✏️ Your turn

Write your code in the empty cells, then run the check cell under each one.

### Task 1: the three smallest

Write `smallest_three(nums)` that returns the three smallest numbers, smallest first.

In [ ]:
import heapq

def smallest_three(nums):
    # your code here
    pass

In [ ]:
from checks import check
check(smallest_three([5, 1, 8, 3, 9, 2]), [1, 2, 3])
check(smallest_three([7, 7, 1, 4]), [1, 4, 7])
check(smallest_three([3, 2, 1]), [1, 2, 3])

### Task 2: sorting with a heap

Write `heap_sort(nums)` that returns a new list with the numbers in order, smallest first. Don't use `sorted` or `sort`: put the numbers in a heap, then pop them all out into a new list.

In [ ]:
def heap_sort(nums):
    # your code here
    pass

In [ ]:
check(heap_sort([3, 1, 2]), [1, 2, 3])
check(heap_sort([5, 5, 1, 9]), [1, 5, 5, 9])
check(heap_sort([]), [])

### Task 3: LeetCode 1046, Last Stone Weight

You have stones with weights. Each turn, take the **two heaviest** stones and smash them together:

- if they weigh the same, both are destroyed,
- if not, the lighter one is destroyed, and the heavier one becomes (heavier − lighter).

Keep going until at most one stone is left. Return its weight, or `0` if none is left.

Example: `[2, 7, 4, 1, 8, 1]`: smash 8 and 7 → 1 is left, so the stones are `[2, 4, 1, 1, 1]`. Smash 4 and 2 → `[2, 1, 1, 1]`. Smash 2 and 1 → `[1, 1, 1]`. Smash 1 and 1 → `[1]`. The answer is `1`.

Hint: you need the heaviest each time. Minus trick.

In [ ]:
class Solution:
    def lastStoneWeight(self, stones: list[int]) -> int:
        # your code here
        pass

In [ ]:
check(Solution().lastStoneWeight([2, 7, 4, 1, 8, 1]), 1)
check(Solution().lastStoneWeight([1]), 1)
check(Solution().lastStoneWeight([2, 2]), 0)
check(Solution().lastStoneWeight([10, 4]), 6)

### Task 4: how many are smaller?

`nums` is sorted. Write `count_smaller(nums, x)` that returns how many items are smaller than `x`. Use `bisect`, not a loop.

In [ ]:
import bisect

def count_smaller(nums, x):
    # your code here
    pass

In [ ]:
check(count_smaller([1, 3, 3, 3, 7], 3), 1)
check(count_smaller([1, 3, 3, 3, 7], 4), 4)
check(count_smaller([1, 3, 3, 3, 7], 0), 0)
check(count_smaller([1, 3, 3, 3, 7], 10), 5)

### Task 5: how many times?

`nums` is sorted. Write `count_of(nums, x)` that returns how many times `x` appears. Use `bisect`.

In [ ]:
def count_of(nums, x):
    # your code here
    pass

In [ ]:
check(count_of([1, 3, 3, 3, 7], 3), 3)
check(count_of([1, 3, 3, 3, 7], 7), 1)
check(count_of([1, 3, 3, 3, 7], 5), 0)
check(count_of([], 5), 0)

### Task 6: LeetCode 35, Search Insert Position

`nums` is sorted and has no repeats. Return the index of `target` if it's there. If it isn't, return the index where it would go to keep the list sorted.

Write the binary search **by hand** this time, with `left`, `right` and `mid`, without `bisect`. Then check your answer against `bisect.bisect_left` if you like.

In [ ]:
class Solution:
    def searchInsert(self, nums: list[int], target: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().searchInsert([1, 3, 5, 6], 5), 2)
check(Solution().searchInsert([1, 3, 5, 6], 2), 1)
check(Solution().searchInsert([1, 3, 5, 6], 7), 4)
check(Solution().searchInsert([1, 3, 5, 6], 0), 0)
check(Solution().searchInsert([4], 4), 0)

### Task 7 (a stretch): LeetCode 703, Kth Largest Element in a Stream

Design a class `KthLargest`:

- `KthLargest(k, nums)` starts with the numbers in `nums`.
- `add(val)` adds `val`, then returns the **k-th largest** number so far.

Example: `KthLargest(3, [4, 5, 8, 2])`, then `add(3)` → 4 (the numbers are now 2, 3, 4, 5, 8, and the 3rd largest is 4), `add(5)` → 5, `add(10)` → 5, `add(9)` → 8, `add(4)` → 8.

Hint: the top-3 leaderboard from above, kept on `self`.

In [ ]:
class KthLargest:
    def __init__(self, k: int, nums: list[int]):
        # your code here
        pass

    def add(self, val: int) -> int:
        # your code here
        pass

In [ ]:
kl = KthLargest(3, [4, 5, 8, 2])
check(kl.add(3), 4)
check(kl.add(5), 5)
check(kl.add(10), 5)
check(kl.add(9), 8)
check(kl.add(4), 8)

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the check cell under it.

**Question 1.** You push 6, 2, 9 and 4 into an empty heap. What is `heap[0]`?

- A) `6`
- B) `9`
- C) `2`

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("22-q1", answer_1)

**Question 2.** How do you get the **biggest** number out first with `heapq`?

- A) Push `-n` instead of `n`, and put the minus back when you pop
- B) Read `heap[-1]`

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("22-q2", answer_2)

**Question 3.** `nums = [1, 3, 3, 3, 7]`. What does `bisect.bisect_right(nums, 3)` give?

- A) `1`
- B) `4`
- C) `3`

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("22-q3", answer_3)

**Question 4.** In a binary search, why is it `mid = (left + right) // 2` and not `/ 2`?

- A) `//` is faster
- B) An index must be a whole number, and `/` gives a decimal

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("22-q4", answer_4)

---

## Done

You can now use a heap to always get the smallest (or, with the minus trick, the biggest), keep the k biggest as numbers arrive, and find a place in a sorted list with `bisect` or with your own binary search.

Stuck on a task? The answers are in `solutions/22-heapq-and-bisect.ipynb`. Try for real first.

**That's the last notebook.** You now know every Python tool the easy LeetCode problems need. Next: keep practising with the problem list (`leetcode-list.pdf`), a few problems a day.